# Data Train/Val/Test Splitting

## Overview
Two notebooks that together prepare all model-ready splits and verify the temporal structure of non-daily features. Notebook 01 is a full pipeline script; Notebook 02 runs after it using data already in memory (or reloaded from saved splits).

---

## Notebook 01: Prepare Datasets (`01_prepare_datasets.ipynb`)

### Purpose
Takes the Stage 4 (calendar-theme-removed) model-ready tables and produces four temporal train/val/test splits × two feature sets (full_moments, means_only), with all associated data quality fixes applied. Also fixes a pandas CSV float-truncation bug in the subtheme ID scheme, re-z-scores weekly features correctly, builds binary and continuous targets, clips z-scores, and writes corrected theme CSVs alongside the split parquets.

### Configuration
Key parameters defined at the top of the script:

- **`TARGET_HORIZON = 5`:** target is the minimum return over the next 5 trading days
- **`CRASH_THRESHOLD = -0.02`:** binary target fires when `minret_5d < -2%`
- **`EMBARGO_ROWS = 5`:** last 5 rows of train and validation are dropped to prevent leakage across the train/val and val/test boundaries
- **`CLIP_LIMIT = 5.0`:** z-scores clipped to ±5 (binary and meta columns excluded)
- **`Z_MIN_WINDOW = 252`, `Z_SHIFT = 6`:** expanding z-score parameters for the continuous target
- **`WEEKLY_Z_MIN_PERIODS = 52`:** minimum 52 weekly observations before first valid weekly z-score

### Four Temporal Splits

| Split | Description | Test Period |
|---|---|---|
| Split_A | Primary: rate hiking bear market | 2022--2023 |
| Split_B | Pre-COVID stress (Volmageddon, Q4 2018) | 2018--2019 |
| Split_C | Black swan + recovery (COVID crash) | 2020--2021 |
| Split_D | Recovery + AI rally (most training data) | 2023--2024 |

Each split has a 2-year validation window immediately preceding the test period, with the remaining history as training data.

### Two Feature Sets
- **`full_moments`:** `model_market_combined_full_moments.parquet` -- cwmean, cwstd, cwskew, cwkurt, spread per stock factor + macro factors (~2,200 features)
- **`means_only`:** `model_market_combined_means.parquet` -- cwmean only + macro factors (~725 features)

---

### Phase 0: Fix Subtheme IDs in Theme CSVs

**Problem:** When subtheme IDs like `'1.10'` are saved to CSV, pandas parses them back as the float `1.1`, which collides with the legitimate subtheme `'1.1'`. Six subthemes are affected: `1.10`, `2.10`, `3.10`, `8.10`, `9.10`, `12.10`.

**Fix (`fix_subtheme_ids`):**
1. Convert `subtheme_id` column from float to string
2. For each of the 6 conflated pairs, identify the correct features by exact column name (using `_expand_base_factors_to_all_column_names` to generate all base names + moment-suffixed variants) and reassign to underscore-format IDs (e.g. `1_10`)
3. Convert **all** remaining dot-format IDs to underscore format (`1.1` → `1_1`, `3.12` → `3_12`) to prevent recurrence

**Additional theme fixes applied within `fix_subtheme_ids`:**
- **`fix_2_11_split`:** moves 10 monthly VolumeTrend features (`monthly_VolumeTrend` and `monthly_volumetrend_chg_1m`, all moment variants) from subtheme `2_8` (daily Volume Dynamics) into new subtheme `2_11` (Monthly Volume Dynamics), giving each subtheme a single clean update frequency
- **`fix_12_11_split`:** moves `initial_claims` and `continued_claims` from subtheme `12_1` (monthly Employment & Labour) into new subtheme `12_11` (Weekly Unemployment Claims) for the same reason

**Validation (`validate_subtheme_fix`):** for each of the 6 conflated pairs, lists all features in both the parent (`X_1`) and restored child (`X_10`) subthemes, verifies no child features leaked back into the parent, and confirms no dot-format IDs remain anywhere.

Fixed CSVs are saved to `Output_DIR/themes/` and used for all subsequent steps.

### Phase 1: Z-Scored Continuous Target Construction

The continuous target is the expanding-window z-score of `minret_5d` (minimum return over the next 5 days). Built from Stage 2 data (which has a longer history than Stage 4) to maximise the expanding window:

`minret_5d_z_t = (minret_5d_t - μ_{1:t-Z_SHIFT}) / σ_{1:t-Z_SHIFT}`

`shift = 6` (6-day shift rather than 1) is used to allow for the fact that `minret_5d` itself looks forward 5 days.

### Phase 1.5: Weekly Feature Z-Score Fix

**Problem:** In the original Stage 3 pipeline, weekly features (CFTC, AAII, FRED weekly) were forward-filled to daily frequency *first*, then z-scored at daily frequency. This caused the z-score to drift slightly every day even when no new weekly data had arrived -- the expanding mean and std were updated by repeated identical values.

**Fix (`build_weekly_zscored_daily`):** for each of the 34 weekly features:
1. Load the raw forward-filled daily values from Panel C engineered
2. Detect genuine update days (where the value changes by more than 1e-12)
3. Compute an expanding z-score at **weekly frequency** using only update-day observations -- strictly causal: z_k uses observations 0..k-1 with minimum 52 periods
4. Forward-fill the weekly z-scores back to daily frequency -- the result is flat between updates

**1-day delay for H.4.1 + Claims:** `fed_assets`, `tga`, `reserves` (Federal Reserve H.4.1, published Thursday) and `initial_claims`, `continued_claims` (DoL, published Thursday) are shifted forward by 1 day so they fire on Friday, aligning them with `bank_credit` and `ci_loans` (Fed H.8, published Friday of the following week). This makes all five features in subtheme `9_7` (Fed Balance Sheet & Banking) update on the same day.

Flatness is verified: weekly features should be flat on ~80% of trading days.

### Phase 2: Process Each Feature Set

For each feature set (full_moments, means_only):

1. **Load** the Stage 4 parquet and fixed theme CSV
2. **Replace weekly features** with correctly z-scored versions (Phase 1.5)
3. **Clip features** to ±5.0 using `clip_zscores`. Binary features (`vix_above_20`, `vix_above_30`, `curve_inverted_2y10y`, `curve_inverted_3m10y`, `credit_stress`) and meta columns are excluded. Reports cells clipped, percentage, and worst pre-clip z-score.
4. **Construct binary target** (`y_binary = (minret_5d < -0.02).astype(float)`) and `minret_5d` from `target_daily_return`
5. **Merge z-scored continuous target** with cross-check: Stage 2 `minret_5d` is recomputed independently and compared to Stage 4 `minret_5d` -- max absolute difference must be < 1e-8
6. **Spot checks** on `minret_5d` correctness at rows 0, 100, 500, 1000
7. **Create all four splits** using `make_split`
8. **Validate each split** and save to parquet

### Splitting Logic (`make_split`)

- Train/val/test assigned by date range
- **Embargo:** last `EMBARGO_ROWS = 5` rows are dropped from train and validation to prevent leakage at the train/val and val/test boundaries (5 days matches `TARGET_HORIZON`)
- Rows with NaN in `y_binary` or `minret_5d_z` are dropped (last few rows where 5-day forward window is incomplete)
- Meta columns (`date`, `target_daily_return`, `minret_5d`, `minret_5d_z`, `y_binary`) prepended to feature columns in the output

### Split Validation (`validate_split`)

- No date overlap between train/val and val/test
- Temporal order: train end < val start, val end < test start
- Zero NaN in all three partitions
- All clipped columns respect ±5.0 limit
- Crash rate, date ranges, and z-score statistics printed for each partition

### Phase 3: Save Metadata

A `metadata.json` is saved with full configuration (target definition, embargo size, clip limit, subtheme ID format, corrections applied) and per-split statistics (row counts, date ranges, crash rates, z-score min/max/std).

### Loader Utilities

Two helper functions are provided for downstream model code:
- **`load_split(split_name, feature_set, base_dir)`:** loads train/val/test parquets and returns numpy arrays `X_train`, `y_train`, `minret_train`, etc. plus feature column names and metadata
- **`load_theme_assignment(feature_set, base_dir)`:** loads the corrected theme assignment CSV

### Outputs
Saved to `Data/Splits/`:
- `Split_A/` through `Split_D/`: each containing `full_moments_train.parquet`, `full_moments_val.parquet`, `full_moments_test.parquet`, `means_only_train.parquet`, `means_only_val.parquet`, `means_only_test.parquet` (24 parquet files total)
- `themes/combined_full_moments_theme_assignment.csv` -- corrected, underscore-format subtheme IDs
- `themes/combined_means_theme_assignment.csv` -- corrected, underscore-format subtheme IDs
- `metadata.json` -- full pipeline configuration and split statistics

---

## Notebook 02: Subtheme Update Schedule & Alignment Verification (`02_subtheme_update_schedule.ipynb`)

### Purpose
Runs after Notebook 01 (uses `df`, `theme_df`, `feature_cols` from memory or reloaded splits). Verifies that within each non-daily subtheme, all features update on exactly the same days as their representative. Produces an update mask DataFrame and summary CSV used by the sparse KAN to know when to process each subtheme.

### Step 1: Build Change-Day Sets
For every feature in `feature_cols`, the set of row indices where the value changes by more than 1e-12 is computed. Days where either the current or previous value is NaN are skipped.

### Step 2: Build Subtheme Mapping with Frequency
Features are grouped by `subtheme_id` from the theme CSV. Only non-daily subthemes (weekly and monthly) are processed further.

### Step 3: Verify Alignment and Build Update Schedules
For each non-daily subtheme:
- **Representative** = feature with the most change-days (most update events)
- **Alignment check:** every other feature's change-days must be a **subset** of the representative's change-days. If a feature changed on a day the representative did not, this is a genuine misalignment (not just staleness) and is flagged as a violation
- **Update pattern** is computed: weekly subthemes show the modal weekday; monthly subthemes show "month-end (day X--Y)" with actual day-of-month range; median gap between updates (in trading days) is also computed

### Step 4: Print Alignment Results
Reports how many non-daily subthemes were fully aligned vs had violations. All violations are listed with the offending feature, number of orphan days, and example dates.

### Step 5: Build and Save Update Mask
A boolean DataFrame of shape `(n_dates × n_non_daily_subthemes)` is constructed. Each column `update_{sid}` is `True` on days when that subtheme received new data (i.e., the representative's value changed), `False` otherwise. Daily subthemes are not included -- they are assumed to always update.

### Step 6: Verify Mask Correctness
Zero-tolerance cross-check: for every non-daily subtheme, every `mask=True` day must have at least one feature that changed, and every `mask=False` day must have zero features that changed.

### Step 7: Sanity Checks
Update rates are checked against expected ranges: weekly subthemes should update 15--25% of trading days (~once per 5 days), monthly subthemes 3--6% (~once per 21 days).

### Outputs
Saved to `Data/Splits/`:
- `subtheme_update_mask.parquet` -- boolean mask `(n_dates × n_non_daily_subthemes)`, loaded by the sparse KAN to know when to process each subtheme
- `subtheme_update_schedule.csv` -- one row per non-daily subtheme: `subtheme_id`, `subtheme_name`, `frequency`, `n_features`, `n_update_days`, `representative_feature`, `update_pattern`, `median_gap_days`

In [5]:
"""
01_prepare_datasets.ipynb
===========================
Dataset Preparation: Four Temporal Splits x Two Feature Sets
Fixes subtheme ID float truncation, clips z-scores to +/-5, splits chronologically.

Includes fix for pandas CSV float truncation bug:
    When saved to CSV, subtheme_id '1.10' is parsed as float 1.1, colliding
    with the real '1.1'. This script identifies the 6 affected subtheme pairs
    by matching exact feature names from the original assignment code (doc 36),
    reassigns them to underscore-format IDs (e.g., '1_10'), and converts ALL
    subtheme IDs to underscore format to prevent recurrence.

Inputs:
    - model_market_combined_full_moments.parquet
    - model_market_combined_means.parquet
    - themes/combined_full_moments_theme_assignment.csv
    - themes/combined_means_theme_assignment.csv

Outputs (saved to OUTPUT_DIR):
    Split_A/ through Split_D/
        full_moments_{train,val,test}.parquet
        means_only_{train,val,test}.parquet
    themes/  (corrected theme CSVs with underscore-format subtheme IDs)
    metadata.json
"""

import pandas as pd
import numpy as np
import json
import shutil
from pathlib import Path
from datetime import datetime


# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

PROJECT_ROOT = Path("../..")
DATA_DIR = PROJECT_ROOT / "Data" / "Data_Collection" / "Final" / "Stage_4_Final_w_Calendar_Theme_Removed"
STAGE2_PATH = PROJECT_ROOT / "Data" / "Data_Collection" / "Final" / "Stage_2" / "agg_market_daily_full_moments.parquet"
OUTPUT_DIR = PROJECT_ROOT / "Data" / "Splits"

TARGET_HORIZON = 5
CRASH_THRESHOLD = -0.02
EMBARGO_ROWS = 5
CLIP_LIMIT = 5.0
Z_MIN_WINDOW = 252
Z_SHIFT = 6

META_COLS = {"date", "target_daily_return", "minret_5d", "minret_5d_z", "y_binary"}

BINARY_FEATURES = [
    'vix_above_20', 'vix_above_30',
    'curve_inverted_2y10y', 'curve_inverted_3m10y',
    'credit_stress',
]

DO_NOT_CLIP = META_COLS | set(BINARY_FEATURES)

MOMENT_SUFFIXES = ['_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread']

# ── Panel C path (raw weekly values before z-scoring) ────────────────────────
PANEL_C_PATH = PROJECT_ROOT / "Data" / "Data_Collection" / "Final" / \
    "Stage_1_5_Validation_and_Feature_Engineering" / "panel_macro_daily_engineered.parquet"

WEEKLY_Z_MIN_PERIODS = 52

# H.4.1 + Claims features to delay by 1 day (Thu -> Fri) to align with
# H.8 (bank_credit, ci_loans) within subtheme 9_7 Fed Balance Sheet & Banking.
# After delay, all 5 features in 9_7 fire on the same day (Friday).
H41_CLAIMS_FEATURES = ['fed_assets', 'tga', 'reserves']  # delay Thu -> Fri to align with H.8 in 9_7

# All 34 weekly-sourced features (used by build_weekly_zscored_daily)
WEEKLY_FEATURES = [
    # CFTC positioning (22) -- Mon
    'lev_long', 'lev_short', 'lev_spread',
    'am_long', 'am_short', 'am_spread',
    'dealer_long', 'dealer_short', 'dealer_spread',
    'other_long', 'other_short', 'other_spread',
    'open_interest',
    'lev_net', 'am_net', 'dealer_net',
    'lev_net_pct', 'am_net_pct', 'dealer_net_pct',
    'lev_am_ratio', 'lev_net_chg', 'am_net_chg',
    # AAII sentiment (5) -- Thu
    'bullish', 'neutral', 'bearish', 'bullish_8w_ma', 'bull_bear_spread',
    # DoL claims (2) -- Thu; delayed +1 day to align with H.8
    'initial_claims', 'continued_claims',
    # Fed H.4.1 (3) -- Thu; delayed +1 day to align with H.8
    'fed_assets', 'tga', 'reserves',
    # Fed H.8 (2) -- Fri
    'bank_credit', 'ci_loans',
]
assert len(set(WEEKLY_FEATURES)) == 34, f"Expected 34, got {len(set(WEEKLY_FEATURES))}"

# Features to move from 2_8 -> new 2_11 "Monthly Volume Dynamics"
# Two monthly base factors (VolumeTrend level + 1m change) sitting in
# an otherwise daily subtheme 2_8 "Volume Dynamics".
# Includes both raw base names (combined_means CSV) and moment-suffixed
# names (combined_full_moments CSV).
VOLUME_TREND_FEATURES = [
    # Raw base names -- appear in combined_means CSV
    'monthly_VolumeTrend',
    'monthly_volumetrend_chg_1m',
    # Moment-suffixed names -- appear in combined_full_moments CSV
    'monthly_VolumeTrend_cwmean', 'monthly_VolumeTrend_cwstd',
    'monthly_VolumeTrend_cwskew', 'monthly_VolumeTrend_cwkurt',
    'monthly_VolumeTrend_spread',
    'monthly_volumetrend_chg_1m_cwmean', 'monthly_volumetrend_chg_1m_cwstd',
    'monthly_volumetrend_chg_1m_cwskew', 'monthly_volumetrend_chg_1m_cwkurt',
    'monthly_volumetrend_chg_1m_spread',
]

# Claims features to move from 12_1 -> 12_11 in theme CSVs.
# initial_claims and continued_claims are weekly; all other 12_1 features are
# monthly. Separating them gives subtheme 12_1 a clean monthly frequency and
# creates new subtheme 12_11 "Weekly Unemployment Claims" with weekly frequency.
CLAIMS_FEATURES = ['initial_claims', 'continued_claims']

# ── Subtheme ID corrections ─────────────────────────────────────────────────
# These 6 subthemes had IDs like '1.10' which pandas truncated to 1.1 (float),
# colliding with the real subtheme '1.1'. The base_factors lists are copied
# verbatim from the original assign() calls in document 36.
SUBTHEME_CORRECTIONS = {
    '1.10': {
        'correct_id': '1_10',
        'correct_name': 'Monthly Liquidity',
        'theme_id': 1,
        'theme_name': 'Liquidity & Market Quality',
        'base_factors': [
            'monthly_BidAskSpread', 'monthly_DolVol', 'monthly_bidask_3m_avg',
            'monthly_ps_level', 'monthly_ps_innov',
        ],
    },
    '2.10': {
        'correct_id': '2_10',
        'correct_name': 'Trade Size & Timing',
        'theme_id': 2,
        'theme_name': 'Order Flow & Participation',
        'base_factors': [
            'csize_to_shrout', 'osize_to_shrout',
            'size_1pm_to_shrout', 'size_4pm_to_shrout',
        ],
    },
    '3.10': {
        'correct_id': '3_10',
        'correct_name': 'VIX Futures & Term Structure',
        'theme_id': 3,
        'theme_name': 'Volatility & Options',
        'base_factors': [
            'vix_fut_front', 'vix_fut_second',
            'vix_term_spread', 'vix_term_ratio', 'vix_futures_basis',
            'vix_fut_ret_1d', 'vix_term_spread_5d_chg',
            'vix_fut_volume', 'vix_fut_oi',
        ],
    },
    '8.10': {
        'correct_id': '8_10',
        'correct_name': 'Retail Sentiment Survey',
        'theme_id': 8,
        'theme_name': 'Analyst Expectations & Sentiment',
        'base_factors': [
            'bullish', 'neutral', 'bearish', 'bullish_8w_ma', 'bull_bear_spread',
        ],
    },
    '9.10': {
        'correct_id': '9_10',
        'correct_name': 'Bond Term Premium',
        'theme_id': 9,
        'theme_name': 'Interest Rates & Monetary Policy',
        'base_factors': [
            'monthly_bond_30y_2y_spread_ret', 'monthly_bond_10y_tbill_spread_ret',
            'monthly_bond_30y_10y_spread_ret', 'monthly_bond_5y_2y_spread_ret',
        ],
    },
    '12.10': {
        'correct_id': '12_10',
        'correct_name': 'Surveys & Leading Indicators',
        'theme_id': 12,
        'theme_name': 'Macroeconomic Fundamentals',
        'base_factors': [
            'monthly_ism_manuf', 'monthly_ism_new_orders',
            'monthly_philly_fed', 'monthly_empire_state',
            'monthly_kansas_fed', 'monthly_chicago_fed_nai',
        ],
    },
}

SPLITS = {
    "Split_B": {
        "description": "Pre-COVID stress (Volmageddon, Q4 2018 selloff)",
        "train_end": "2015-12-31", "val_start": "2016-01-01",
        "val_end": "2017-12-31", "test_start": "2018-01-01", "test_end": "2019-12-31",
    },
    "Split_C": {
        "description": "Black swan + recovery (COVID crash and rebound)",
        "train_end": "2017-12-31", "val_start": "2018-01-01",
        "val_end": "2019-12-31", "test_start": "2020-01-01", "test_end": "2021-12-31",
    },
    "Split_A": {
        "description": "Primary: rate hiking bear market",
        "train_end": "2019-12-31", "val_start": "2020-01-01",
        "val_end": "2021-12-31", "test_start": "2022-01-01", "test_end": "2023-12-31",
    },
    "Split_D": {
        "description": "Recovery + AI rally (most training data)",
        "train_end": "2020-12-31", "val_start": "2021-01-01",
        "val_end": "2022-12-31", "test_start": "2023-01-01", "test_end": "2024-12-31",
    },
}

FEATURE_SETS = {
    "full_moments": {
        "source_file": "model_market_combined_full_moments.parquet",
        "theme_file": "themes/combined_full_moments_theme_assignment.csv",
    },
    "means_only": {
        "source_file": "model_market_combined_means.parquet",
        "theme_file": "themes/combined_means_theme_assignment.csv",
    },
}


# ═══════════════════════════════════════════════════════════════════════════════
# SUBTHEME ID FIX
# ═══════════════════════════════════════════════════════════════════════════════

def _expand_base_factors_to_all_column_names(base_factors: list) -> set:
    """
    Given a list of base factor names, return the set of ALL possible
    column names they could appear as in either the means or full moments file.

    For means files: columns are just the base factor names.
    For full moments files: stock-level factors also appear with moment suffixes.
    Macro factors appear as raw names only.

    We generate ALL possibilities (base + suffixed) and let the matching
    filter to what actually exists. No false positives because factor names
    are unique across the entire pipeline.
    """
    names = set(base_factors)
    for bf in base_factors:
        for suffix in MOMENT_SUFFIXES:
            names.add(f"{bf}{suffix}")
    return names


def fix_subtheme_ids(theme_df: pd.DataFrame, csv_name: str) -> pd.DataFrame:
    """
    Fix float-truncated subtheme IDs and convert ALL IDs to underscore format.

    Steps:
        1. Convert subtheme_id from float to string (prevents further truncation)
        2. For each of the 6 conflated pairs, identify features by their exact
           column names and reassign to the correct subtheme ID/name
        3. Convert ALL remaining dot-format IDs to underscore format
           ('1.1' -> '1_1', '3.12' -> '3_12', etc.)

    Returns the fixed DataFrame. Prints detailed log of changes.
    """
    df = theme_df.copy()

    # Step 1: convert to string
    df['subtheme_id'] = df['subtheme_id'].astype(str)

    print(f"\n  Fixing subtheme IDs in: {csv_name}")
    print(f"  Subthemes before fix: {df['subtheme_id'].nunique()}")

    # Step 2: fix the 6 conflated pairs
    total_fixed = 0
    for orig_id, info in SUBTHEME_CORRECTIONS.items():
        all_names = _expand_base_factors_to_all_column_names(info['base_factors'])
        mask = df['column'].isin(all_names)
        n_matched = mask.sum()

        if n_matched > 0:
            df.loc[mask, 'subtheme_id'] = info['correct_id']
            df.loc[mask, 'subtheme_name'] = info['correct_name']
            # Also fix theme_id/theme_name in case they're present and wrong
            if 'theme_id' in df.columns:
                df.loc[mask, 'theme_id'] = info['theme_id']
            if 'theme_name' in df.columns:
                df.loc[mask, 'theme_name'] = info['theme_name']
            total_fixed += n_matched

        print(f"    {orig_id:>5} -> {info['correct_id']:<5} "
              f"({info['correct_name']:<30}) : {n_matched} features matched")

    # Step 3: convert ALL dot-format IDs to underscore format
    # This prevents any future float truncation when the CSV is re-read
    # Already-fixed IDs use underscores so this replace is a no-op for them
    df['subtheme_id'] = df['subtheme_id'].str.replace('.', '_', regex=False)

    print(f"  Total features reassigned: {total_fixed}")
    print(f"  Subthemes after fix: {df['subtheme_id'].nunique()}")
    print(f"  All IDs converted to underscore format")

    # Apply 2_11 split: move monthly VolumeTrend features out of 2_8 into new 2_11
    df = fix_2_11_split(df, csv_name)

    # Apply 12_11 split: move claims features out of 12_1 into new 12_11
    df = fix_12_11_split(df, csv_name)

    return df


def validate_subtheme_fix(theme_df: pd.DataFrame, csv_name: str) -> bool:
    """
    Validate the subtheme fix by printing every feature in the 6 conflated
    pairs (both the parent X_1 and the restored X_10) so the user can
    visually confirm no features were left behind or incorrectly moved.

    Returns True if all checks pass.
    """
    errors = []
    print(f"\n  {'=' * 70}")
    print(f"  SUBTHEME FIX VALIDATION: {csv_name}")
    print(f"  {'=' * 70}")

    # For each conflated pair, show what's in X_1 and what's in X_10
    PAIRS = [
        ('1_1',  'CRSP Bid-Ask Spread & Dynamics',  '1_10', 'Monthly Liquidity'),
        ('2_1',  'Aggregate Buy-Sell Balance',       '2_10', 'Trade Size & Timing'),
        ('3_1',  'Stock Implied Volatility Levels',  '3_10', 'VIX Futures & Term Structure'),
        ('8_1',  'Price Target Consensus & Disagreement', '8_10', 'Retail Sentiment Survey'),
        ('9_1',  'Yield Curve Level',                '9_10', 'Bond Term Premium'),
        ('12_1', 'Employment & Labour',              '12_10', 'Surveys & Leading Indicators'),
    ]

    for parent_id, parent_name, child_id, child_name in PAIRS:
        parent_rows = theme_df[theme_df['subtheme_id'] == parent_id]
        child_rows = theme_df[theme_df['subtheme_id'] == child_id]

        print(f"\n  --- Pair: {parent_id} / {child_id} ---")

        print(f"\n  Subtheme {parent_id} ({parent_name}): {len(parent_rows)} features")
        if len(parent_rows) > 0:
            for _, row in parent_rows.iterrows():
                print(f"    {row['column']}")
        else:
            print(f"    (empty)")

        print(f"\n  Subtheme {child_id} ({child_name}): {len(child_rows)} features")
        if len(child_rows) > 0:
            for _, row in child_rows.iterrows():
                print(f"    {row['column']}")
        else:
            errors.append(f"{child_id} ({child_name}) has ZERO features -- fix failed?")
            print(f"    ✗ EMPTY -- fix may have failed!")

        # Cross-check: verify no child features remain in parent
        child_base_factors = SUBTHEME_CORRECTIONS.get(
            child_id.replace('_', '.'), {}
        ).get('base_factors', [])
        child_all_names = _expand_base_factors_to_all_column_names(child_base_factors)

        leaked = parent_rows[parent_rows['column'].isin(child_all_names)]
        if len(leaked) > 0:
            errors.append(f"{len(leaked)} features that belong in {child_id} "
                          f"are still in {parent_id}: {leaked['column'].tolist()}")
            print(f"    ✗ LEAK: {leaked['column'].tolist()} should be in {child_id}!")

    # Summary
    n_subthemes = theme_df['subtheme_id'].nunique()
    print(f"\n  Total subthemes: {n_subthemes}")

    # Check no dots remain in any subtheme_id
    has_dots = theme_df['subtheme_id'].str.contains(r'\.', regex=True)
    if has_dots.any():
        bad = theme_df.loc[has_dots, 'subtheme_id'].unique().tolist()
        errors.append(f"Dot-format IDs still present: {bad}")

    if errors:
        print(f"\n  ✗ VALIDATION FAILED:")
        for e in errors:
            print(f"    {e}")
        return False
    else:
        print(f"\n  ✓ All {n_subthemes} subthemes validated, "
              f"zero leaks, zero dot-format IDs")
        return True


# ═══════════════════════════════════════════════════════════════════════════════
# THEME CSV FIXES: 9_7 ALIGNMENT + 12_1 SPLIT
# ═══════════════════════════════════════════════════════════════════════════════

def fix_2_11_split(theme_df: pd.DataFrame, csv_name: str) -> pd.DataFrame:
    """
    Move 10 monthly VolumeTrend features from subtheme 2_8 to new
    subtheme 2_11 "Monthly Volume Dynamics".

    2_8 is a daily subtheme (Volume Dynamics) containing 25 daily features
    plus 10 monthly features (VolumeTrend level + 1m change, 5 moments each).
    Moving the monthly features to 2_11 gives every subtheme a single
    clean update frequency.
    """
    df = theme_df.copy()

    mask = df['column'].isin(VOLUME_TREND_FEATURES)
    n_matched = mask.sum()

    if n_matched == 0:
        print(f"    WARNING: no VolumeTrend features found in {csv_name}")
        return df

    df.loc[mask, 'subtheme_id']   = '2_11'
    df.loc[mask, 'subtheme_name'] = 'Monthly Volume Dynamics'
    if 'theme_id'   in df.columns: df.loc[mask, 'theme_id']   = 2
    if 'theme_name' in df.columns: df.loc[mask, 'theme_name'] = 'Order Flow & Participation'

    print(f"    Moved {n_matched} features to 2_11 (Monthly Volume Dynamics)")

    # Verify 2_8 no longer contains monthly VolumeTrend features
    remaining = df[(df['subtheme_id'] == '2_8') & df['column'].isin(VOLUME_TREND_FEATURES)]
    assert len(remaining) == 0, f"VolumeTrend features still in 2_8: {remaining['column'].tolist()}"

    new_sub = df[df['subtheme_id'] == '2_11']
    assert len(new_sub) == n_matched, f"2_11 has {len(new_sub)} features, expected {n_matched}"
    print(f"    ✓ 2_8 now has {(df['subtheme_id'] == '2_8').sum()} features (daily only)")
    print(f"    ✓ 2_11 created with {len(new_sub)} features (monthly only)")

    return df


def fix_12_11_split(theme_df: pd.DataFrame, csv_name: str) -> pd.DataFrame:
    """
    Move initial_claims and continued_claims from subtheme 12_1 to new
    subtheme 12_11 "Weekly Unemployment Claims".

    12_1 contains monthly employment features + 2 weekly claims features.
    Moving the claims features to 12_11 gives every subtheme a single
    clean update frequency.
    """
    df = theme_df.copy()

    mask = df['column'].isin(CLAIMS_FEATURES)
    n_matched = mask.sum()

    if n_matched == 0:
        print(f"    WARNING: no claims features found in {csv_name}")
        return df

    df.loc[mask, 'subtheme_id']   = '12_11'
    df.loc[mask, 'subtheme_name'] = 'Weekly Unemployment Claims'
    if 'theme_id'   in df.columns: df.loc[mask, 'theme_id']   = 12
    if 'theme_name' in df.columns: df.loc[mask, 'theme_name'] = 'Macroeconomic Fundamentals'

    print(f"    Moved {n_matched} features to 12_11 (Weekly Unemployment Claims): "
          f"{CLAIMS_FEATURES}")

    # Verify 12_1 no longer contains claims
    remaining = df[(df['subtheme_id'] == '12_1') & df['column'].isin(CLAIMS_FEATURES)]
    assert len(remaining) == 0, f"Claims features still in 12_1: {remaining['column'].tolist()}"

    # Verify 12_11 now exists
    new_sub = df[df['subtheme_id'] == '12_11']
    assert len(new_sub) == n_matched, f"12_11 has {len(new_sub)} features, expected {n_matched}"
    print(f"    ✓ 12_1 now has {(df['subtheme_id'] == '12_1').sum()} features (monthly only)")
    print(f"    ✓ 12_11 created with {len(new_sub)} features (weekly only)")

    return df


# ═══════════════════════════════════════════════════════════════════════════════
# WEEKLY FEATURE Z-SCORE FIX
# ═══════════════════════════════════════════════════════════════════════════════

def build_weekly_zscored_daily(
    panel_c_path: Path,
    weekly_features: list,
    min_periods: int = WEEKLY_Z_MIN_PERIODS,
) -> pd.DataFrame:
    """
    Build correctly z-scored weekly features at daily frequency.

    Problem: in the original pipeline, weekly features were forward-filled to
    daily FIRST, then z-scored at daily frequency. This caused the z-score to
    drift every day even when no new weekly data had arrived.

    Fix:
        1. Load raw forward-filled daily values from Panel C engineered.
        2. Detect genuine update days (where value changes from previous day).
        3. Z-score at WEEKLY frequency using an expanding window on update-day
           observations only. Strictly causal: z_k uses observations 0..k-1.
        4. Forward-fill z-scores back to daily. Result is flat between updates.
        5. Delay H.4.1+Claims (fed_assets, tga, reserves, initial_claims,
           continued_claims) by 1 day to align with H.8 (bank_credit, ci_loans)
           within subtheme 9_7. All 5 features in 9_7 then fire on Friday.
    """
    cols_needed = ['date'] + weekly_features
    panel_c = pd.read_parquet(panel_c_path, columns=cols_needed)
    panel_c['date'] = pd.to_datetime(panel_c['date'])
    panel_c = panel_c.sort_values('date').reset_index(drop=True)

    n_days = len(panel_c)
    result = pd.DataFrame({'date': panel_c['date']})

    print(f"\n  Building weekly-frequency z-scores from Panel C engineered...")
    print(f"    Source rows: {n_days:,}  "
          f"({panel_c['date'].min().date()} -> {panel_c['date'].max().date()})")
    print(f"    Weekly features: {len(weekly_features)}")
    print(f"    Min periods: {min_periods}")
    print(f"    1-day delay applied to: {H41_CLAIMS_FEATURES}")

    stats_log = []

    for col_name in weekly_features:
        raw = panel_c[col_name].values.copy().astype(np.float64)

        # Step 1: detect genuine update days
        is_update = np.zeros(n_days, dtype=bool)
        is_update[0] = True
        for i in range(1, n_days):
            if abs(raw[i] - raw[i - 1]) > 1e-12:
                is_update[i] = True

        update_idx  = np.where(is_update)[0]
        update_vals = raw[update_idx]
        n_updates   = len(update_vals)

        # Step 2: expanding z-score at weekly frequency (strictly causal)
        z_at_updates = np.full(n_updates, np.nan)
        for k in range(min_periods, n_updates):
            past  = update_vals[:k]
            mu    = np.mean(past)
            sigma = np.std(past, ddof=1)
            if sigma > 1e-12:
                z_at_updates[k] = (update_vals[k] - mu) / sigma

        # Step 3: forward-fill z-scores back to daily
        daily_z = np.full(n_days, np.nan)
        for k in range(n_updates):
            start = update_idx[k]
            end   = update_idx[k + 1] if k + 1 < n_updates else n_days
            daily_z[start:end] = z_at_updates[k]

        # Step 4: delay H.4.1+Claims by 1 day (shift forward)
        # This moves their Thursday z-score to Friday, aligning them with
        # H.8 (bank_credit, ci_loans) so all of subtheme 9_7 fires on Friday.
        if col_name in H41_CLAIMS_FEATURES:
            daily_z = np.roll(daily_z, 1)
            daily_z[0] = np.nan  # first row loses its value from the shift

        result[col_name] = daily_z

        n_valid = int(np.sum(~np.isnan(daily_z)))
        first_valid_idx = int(np.argmax(~np.isnan(daily_z))) if n_valid > 0 else -1
        first_valid_dt  = panel_c['date'].iloc[first_valid_idx] if first_valid_idx >= 0 else None
        stats_log.append({
            'col': col_name, 'n_updates': n_updates,
            'n_valid': n_valid, 'first_valid': first_valid_dt,
            'delayed': col_name in H41_CLAIMS_FEATURES,
        })

    # Summary
    print(f"\n    {'Feature':<25} {'Updates':>8} {'Valid days':>11} "
          f"{'First valid':>14} {'Delayed':>8}")
    print(f"    {'-' * 70}")
    for s in stats_log:
        fv = s['first_valid'].date() if s['first_valid'] is not None else 'N/A'
        delayed_str = '+1d' if s['delayed'] else ''
        print(f"    {s['col']:<25} {s['n_updates']:>8,} {s['n_valid']:>11,} "
              f"{str(fv):>14} {delayed_str:>8}")

    return result


def replace_weekly_features(
    df_model_ready: pd.DataFrame,
    weekly_z_daily: pd.DataFrame,
) -> pd.DataFrame:
    """Replace the 34 weekly feature columns with correctly z-scored versions."""
    weekly_cols = [c for c in weekly_z_daily.columns if c != 'date']
    present  = [c for c in weekly_cols if c in df_model_ready.columns]
    missing  = [c for c in weekly_cols if c not in df_model_ready.columns]

    if missing:
        print(f"    WARNING: {len(missing)} weekly features not in model-ready data: "
              f"{missing[:5]}{'...' if len(missing) > 5 else ''}")

    n_before = len(df_model_ready)
    df = df_model_ready.drop(columns=present)
    df = pd.merge(df, weekly_z_daily[['date'] + present], on='date', how='left')
    assert len(df) == n_before, f"Row count changed: {n_before} -> {len(df)}"

    n_nan   = df[present].isna().sum().sum()
    n_cells = len(df) * len(present)
    print(f"    Replaced {len(present)} weekly features with correct z-scores")
    print(f"    NaN introduced (warmup): {n_nan:,} / {n_cells:,} "
          f"({n_nan / n_cells * 100:.2f}%)")
    return df


def validate_weekly_flatness(df: pd.DataFrame, weekly_features: list, n_samples: int = 5):
    """Confirm weekly features are flat between updates (~80% flat days)."""
    present = [c for c in weekly_features if c in df.columns]
    if not present:
        return

    all_changes = 0
    all_total   = 0
    for col in present:
        vals = df[col].dropna().values
        if len(vals) > 1:
            all_changes += int(np.sum(np.abs(np.diff(vals)) > 1e-12))
            all_total   += len(vals) - 1

    if all_total == 0:
        return

    pct_flat = (1 - all_changes / all_total) * 100
    status = "✓" if pct_flat > 75 else "✗ WARNING"
    print(f"    Weekly flatness: {pct_flat:.1f}% flat days (expect ~80%)  {status}")


# ═══════════════════════════════════════════════════════════════════════════════
# Z-SCORE CLIPPING (features only)
# ═══════════════════════════════════════════════════════════════════════════════

def clip_zscores(df: pd.DataFrame, feature_cols: list, limit: float = CLIP_LIMIT) -> dict:
    """Clip z-scored feature columns to +/- limit. Skips binary and meta columns."""
    clip_cols = [c for c in feature_cols if c not in DO_NOT_CLIP and c in df.columns]
    skip_cols = [c for c in feature_cols if c in DO_NOT_CLIP and c in df.columns]

    abs_vals = df[clip_cols].abs()
    max_abs_pre = abs_vals.max().max()
    worst_col = abs_vals.max().idxmax()
    n_beyond = (abs_vals > limit).sum().sum()
    total_cells = len(df) * len(clip_cols)

    binary_pre = {c: df[c].copy() for c in skip_cols if c in BINARY_FEATURES}
    meta_pre = {c: df[c].copy() for c in skip_cols if c in META_COLS and c in df.columns}

    df[clip_cols] = df[clip_cols].clip(lower=-limit, upper=limit)

    for c, pre in binary_pre.items():
        assert (df[c] == pre).all(), f"FAIL: Binary {c} modified"
    for c, pre in meta_pre.items():
        assert df[c].equals(pre), f"FAIL: Meta {c} modified"

    post_max = df[clip_cols].abs().max().max()
    assert post_max <= limit + 1e-10, f"FAIL: Post-clip max = {post_max}"
    assert df[clip_cols].isna().sum().sum() == 0, "FAIL: NaN from clipping"

    return {
        "n_clipped": int(n_beyond), "total_cells": total_cells,
        "pct_clipped": n_beyond / total_cells * 100 if total_cells > 0 else 0,
        "max_abs_pre": float(max_abs_pre), "worst_col": worst_col,
        "post_max": float(post_max),
        "n_clip_cols": len(clip_cols), "n_skip_cols": len(skip_cols),
    }


# ═══════════════════════════════════════════════════════════════════════════════
# TARGET CONSTRUCTION
# ═══════════════════════════════════════════════════════════════════════════════

def construct_binary_target(df: pd.DataFrame) -> pd.DataFrame:
    """Construct minret_5d and y_binary from target_daily_return."""
    ret = df["target_daily_return"].values.copy()
    n = len(ret)
    minret = np.full(n, np.nan)
    for t in range(n - TARGET_HORIZON + 1):
        minret[t] = np.min(ret[t : t + TARGET_HORIZON])

    df = df.copy()
    df["minret_5d"] = minret
    df["y_binary"] = (minret < CRASH_THRESHOLD).astype(float)
    df.loc[df["minret_5d"].isna(), "y_binary"] = np.nan
    return df


def build_z_scored_target() -> pd.DataFrame:
    """Build expanding z-scored continuous target from Stage 2 data."""
    print(f"  Building z-scored continuous target from Stage 2...")

    stg2 = pd.read_parquet(STAGE2_PATH, columns=['date', 'target_daily_return'])
    stg2['date'] = pd.to_datetime(stg2['date'])
    stg2 = stg2.sort_values('date').reset_index(drop=True)
    print(f"    Stage 2 rows: {len(stg2):,}  "
          f"({stg2['date'].min().date()} -> {stg2['date'].max().date()})")

    ret = stg2["target_daily_return"].values.copy()
    n = len(ret)
    minret_raw = np.full(n, np.nan)
    for t in range(n - TARGET_HORIZON + 1):
        minret_raw[t] = np.min(ret[t : t + TARGET_HORIZON])
    stg2["minret_5d_raw"] = minret_raw

    expanding_mean = stg2["minret_5d_raw"].expanding(min_periods=Z_MIN_WINDOW).mean().shift(Z_SHIFT)
    expanding_std = stg2["minret_5d_raw"].expanding(min_periods=Z_MIN_WINDOW).std().shift(Z_SHIFT)
    expanding_std = expanding_std.replace(0, np.nan)
    stg2["minret_5d_z"] = (stg2["minret_5d_raw"] - expanding_mean) / expanding_std

    valid = stg2["minret_5d_z"].dropna()
    print(f"    First valid: {stg2.loc[valid.index[0], 'date'].date()}")
    print(f"    z range: [{valid.min():.2f}, {valid.max():.2f}]")
    print(f"    |z|>5: {(valid.abs() > 5).sum()} days")

    return stg2[['date', 'minret_5d_z']].copy()


def augment_with_z_scored_target(df_stage4: pd.DataFrame, z_df: pd.DataFrame) -> pd.DataFrame:
    """Merge pre-computed z-scored target with cross-check."""
    # Cross-check Stage 2 vs Stage 4 minret_5d
    stg2_raw = pd.read_parquet(STAGE2_PATH, columns=['date', 'target_daily_return'])
    stg2_raw['date'] = pd.to_datetime(stg2_raw['date'])
    ret2 = stg2_raw.sort_values('date')['target_daily_return'].values.copy()
    n2 = len(ret2)
    minret2 = np.full(n2, np.nan)
    for t in range(n2 - TARGET_HORIZON + 1):
        minret2[t] = np.min(ret2[t : t + TARGET_HORIZON])
    stg2_raw = stg2_raw.sort_values('date').reset_index(drop=True)
    stg2_raw['minret_5d_stg2'] = minret2

    cross = pd.merge(
        df_stage4[['date', 'minret_5d']].dropna(subset=['minret_5d']),
        stg2_raw[['date', 'minret_5d_stg2']].dropna(subset=['minret_5d_stg2']),
        on='date', how='inner',
    )
    max_diff = (cross['minret_5d'] - cross['minret_5d_stg2']).abs().max()
    assert max_diff < 1e-8, f"Stage 2/4 minret_5d mismatch: {max_diff:.2e}"
    print(f"    ✓ Cross-check passed (max diff = {max_diff:.2e})")

    df_merged = pd.merge(df_stage4, z_df, on='date', how='left')
    assert len(df_merged) == len(df_stage4), "Merge changed row count"

    valid_z = df_merged["minret_5d_z"].dropna()
    print(f"    ✓ Merged. z range: [{valid_z.min():.2f}, {valid_z.max():.2f}]  "
          f"|z|>5: {(valid_z.abs() > 5).sum()}")

    return df_merged


# ═══════════════════════════════════════════════════════════════════════════════
# SPLITTING LOGIC
# ═══════════════════════════════════════════════════════════════════════════════

def make_split(df, split_config, feature_cols, embargo=EMBARGO_ROWS):
    """Split into train/val/test with embargo and NaN target removal."""
    dates = pd.to_datetime(df["date"])

    train_df = df[dates <= split_config["train_end"]].copy()
    val_df = df[(dates >= split_config["val_start"]) & (dates <= split_config["val_end"])].copy()
    test_df = df[(dates >= split_config["test_start"]) & (dates <= split_config["test_end"])].copy()

    n_train_before = len(train_df)
    if len(train_df) > embargo:
        train_df = train_df.iloc[:-embargo]
    train_embargo = n_train_before - len(train_df)

    n_val_before = len(val_df)
    if len(val_df) > embargo:
        val_df = val_df.iloc[:-embargo]
    val_embargo = n_val_before - len(val_df)

    meta_order = ["date", "target_daily_return", "minret_5d", "minret_5d_z", "y_binary"]
    cols = [c for c in meta_order if c in df.columns] + [c for c in feature_cols if c in df.columns]

    dropna_cols = [c for c in ["y_binary", "minret_5d_z"] if c in cols]

    train_df = train_df[cols].dropna(subset=dropna_cols).reset_index(drop=True)
    val_df = val_df[cols].dropna(subset=dropna_cols).reset_index(drop=True)
    test_df = test_df[cols].dropna(subset=dropna_cols).reset_index(drop=True)

    meta = {}
    for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
        meta[f"{name}_rows"] = len(part)
        meta[f"{name}_date_range"] = [str(part["date"].min()), str(part["date"].max())]
        meta[f"{name}_crash_rate"] = float(part["y_binary"].mean())
        meta[f"{name}_crash_count"] = int(part["y_binary"].sum())
        z = part["minret_5d_z"]
        meta[f"{name}_z_mean"] = float(z.mean())
        meta[f"{name}_z_std"] = float(z.std())
        meta[f"{name}_z_min"] = float(z.min())
        meta[f"{name}_z_max"] = float(z.max())
        meta[f"{name}_z_beyond5"] = int((z.abs() > 5).sum())

    meta["train_embargo_dropped"] = train_embargo
    meta["val_embargo_dropped"] = val_embargo
    meta["n_features"] = len(feature_cols)

    return {"train": train_df, "val": val_df, "test": test_df, "meta": meta}


# ═══════════════════════════════════════════════════════════════════════════════
# SPLIT VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

def validate_split(result, split_name, feature_set_name):
    """Run validation checks on a single split. Returns True if all pass."""
    train_df, val_df, test_df = result["train"], result["val"], result["test"]
    meta = result["meta"]
    errors = []

    print(f"\n  --- {split_name} / {feature_set_name} ---")

    # No overlap
    for a, b, label in [("train", "val", "Train/Val"), ("val", "test", "Val/Test")]:
        da = set(pd.to_datetime(result[a]["date"]))
        db = set(pd.to_datetime(result[b]["date"]))
        if da & db:
            errors.append(f"{label} date overlap ({len(da & db)} dates)")

    # Temporal order
    if pd.to_datetime(train_df["date"]).max() >= pd.to_datetime(val_df["date"]).min():
        errors.append("Train end >= Val start")
    if pd.to_datetime(val_df["date"]).max() >= pd.to_datetime(test_df["date"]).min():
        errors.append("Val end >= Test start")

    # No NaN
    for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
        n_nan = part.drop(columns=["date"]).isna().sum().sum()
        if n_nan > 0:
            errors.append(f"{name}: {n_nan} NaN values")

    # Clipping respected
    for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
        clip_cols = [c for c in part.columns if c not in META_COLS and c not in BINARY_FEATURES]
        if clip_cols:
            mx = part[clip_cols].abs().max().max()
            if mx > CLIP_LIMIT + 1e-10:
                errors.append(f"{name}: max |z| = {mx:.3f} exceeds {CLIP_LIMIT}")

    # Print summary
    print(f"    Train: {meta['train_rows']:>5,} rows  "
          f"({meta['train_date_range'][0][:10]} -> {meta['train_date_range'][1][:10]})  "
          f"crash: {meta['train_crash_rate']:.1%} ({meta['train_crash_count']})")
    print(f"    Val:   {meta['val_rows']:>5,} rows  "
          f"({meta['val_date_range'][0][:10]} -> {meta['val_date_range'][1][:10]})  "
          f"crash: {meta['val_crash_rate']:.1%} ({meta['val_crash_count']})")
    print(f"    Test:  {meta['test_rows']:>5,} rows  "
          f"({meta['test_date_range'][0][:10]} -> {meta['test_date_range'][1][:10]})  "
          f"crash: {meta['test_crash_rate']:.1%} ({meta['test_crash_count']})")

    if errors:
        for e in errors:
            print(f"    ✗ {e}")
        return False
    print(f"    ✓ All checks passed")
    return True


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    print("=" * 70)
    print("DATASET PREPARATION: 4 Splits x 2 Feature Sets")
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    all_metadata = {}
    all_passed = True

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 0: FIX THEME CSVs AND COPY TO OUTPUT
    # ══════════════════════════════════════════════════════════════════════

    print(f"\n{'=' * 70}")
    print("PHASE 0: FIX SUBTHEME IDs IN THEME ASSIGNMENT CSVs")
    print(f"{'=' * 70}")

    themes_dir = OUTPUT_DIR / "themes"
    themes_dir.mkdir(parents=True, exist_ok=True)

    for fs_name, fs_config in FEATURE_SETS.items():
        src = DATA_DIR / fs_config["theme_file"]
        csv_name = Path(fs_config["theme_file"]).name
        dst = themes_dir / csv_name

        if not src.exists():
            print(f"\n  WARNING: {src} not found")
            continue

        theme_df = pd.read_csv(src)
        theme_df = fix_subtheme_ids(theme_df, csv_name)
        ok = validate_subtheme_fix(theme_df, csv_name)
        if not ok:
            all_passed = False

        theme_df.to_csv(dst, index=False)
        print(f"\n  Saved fixed CSV: {dst}")

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 1: BUILD Z-SCORED TARGET (once, shared across feature sets)
    # ══════════════════════════════════════════════════════════════════════

    print(f"\n{'=' * 70}")
    print("PHASE 1: Z-SCORED TARGET CONSTRUCTION")
    print(f"{'=' * 70}")
    z_df = build_z_scored_target()

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 1.5: WEEKLY FEATURE Z-SCORE FIX + 9_7 1-DAY DELAY
    # ══════════════════════════════════════════════════════════════════════
    print(f"\n{'=' * 70}")
    print("PHASE 1.5: WEEKLY FEATURE Z-SCORE FIX + 9_7 1-DAY DELAY")
    print(f"{'=' * 70}")
    weekly_z_daily = build_weekly_zscored_daily(PANEL_C_PATH, WEEKLY_FEATURES)

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 2: PROCESS EACH FEATURE SET
    # ══════════════════════════════════════════════════════════════════════

    for fs_name, fs_config in FEATURE_SETS.items():
        print(f"\n{'=' * 70}")
        print(f"PHASE 2: FEATURE SET — {fs_name}")
        print(f"{'=' * 70}")

        # Load data
        source_path = DATA_DIR / fs_config["source_file"]
        print(f"  Loading: {source_path}")
        df = pd.read_parquet(source_path)
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date").reset_index(drop=True)
        print(f"  Shape: {df.shape[0]:,} x {df.shape[1]}")

        # Load FIXED theme CSV from output dir (not source dir)
        csv_name = Path(fs_config["theme_file"]).name
        theme_path = themes_dir / csv_name
        print(f"  Loading fixed theme CSV: {theme_path}")
        theme_df = pd.read_csv(theme_path)
        feature_cols = [c for c in theme_df["column"].values if c in df.columns]
        print(f"  Features: {len(feature_cols)}")

        # Fix weekly features: replace daily-drifted z-scores with flat ones
        # and apply 1-day delay to H.4.1+Claims to align with H.8 in 9_7.
        # Must happen BEFORE clipping so new z-scores go through the same clip pass.
        print(f"\n  Fixing weekly feature z-scores (9_7 1-day delay included)...")
        df = replace_weekly_features(df, weekly_z_daily)
        validate_weekly_flatness(df, WEEKLY_FEATURES)

        # Clip features
        print(f"\n  Clipping features to +/-{CLIP_LIMIT}...")
        clip_stats = clip_zscores(df, feature_cols)
        print(f"    Pre-clip max: {clip_stats['max_abs_pre']:.1f} ({clip_stats['worst_col']})")
        print(f"    Clipped: {clip_stats['n_clipped']:,} / {clip_stats['total_cells']:,} "
              f"({clip_stats['pct_clipped']:.4f}%)")

        # Binary target
        print(f"\n  Constructing targets...")
        df = construct_binary_target(df)
        n_valid = df["y_binary"].notna().sum()
        n_crash = int(df["y_binary"].sum())
        print(f"  Crash rate: {n_crash / n_valid:.1%} ({n_crash} events)")

        # Z-scored target
        print(f"  Merging z-scored target...")
        df = augment_with_z_scored_target(df, z_df)

        # Spot-check
        ret = df["target_daily_return"].values
        for idx in [0, 100, 500, 1000]:
            if idx + TARGET_HORIZON <= len(ret):
                expected = np.min(ret[idx : idx + TARGET_HORIZON])
                actual = df["minret_5d"].iloc[idx]
                assert abs(expected - actual) < 1e-10, f"Spot check failed at row {idx}"
        print(f"  ✓ Spot checks passed")

        # Create splits
        for split_name, split_config in SPLITS.items():
            split_dir = OUTPUT_DIR / split_name
            split_dir.mkdir(parents=True, exist_ok=True)

            result = make_split(df, split_config, feature_cols)
            ok = validate_split(result, split_name, fs_name)
            if not ok:
                all_passed = False

            for part in ["train", "val", "test"]:
                out = split_dir / f"{fs_name}_{part}.parquet"
                result[part].to_parquet(out, index=False, engine="pyarrow")

            all_metadata[f"{split_name}/{fs_name}"] = {
                **result["meta"],
                "description": split_config["description"],
                "feature_set": fs_name,
                "clip_limit": CLIP_LIMIT,
            }

    # ══════════════════════════════════════════════════════════════════════
    # PHASE 3: SAVE METADATA AND REPORT
    # ══════════════════════════════════════════════════════════════════════

    meta_path = OUTPUT_DIR / "metadata.json"
    full_meta = {
        "created": datetime.now().isoformat(),
        "target_binary": f"y_binary = (minret_{TARGET_HORIZON}d < {CRASH_THRESHOLD})",
        "target_continuous": f"minret_5d_z = expanding z-score (min={Z_MIN_WINDOW}, shift={Z_SHIFT})",
        "embargo_rows": EMBARGO_ROWS, "clip_limit": CLIP_LIMIT,
        "subtheme_id_format": "underscore (e.g., 1_1, 1_10, 3_12)",
        "subtheme_corrections_applied": list(SUBTHEME_CORRECTIONS.keys()),
        "meta_cols": sorted(META_COLS),
        "splits": all_metadata,
    }
    with open(meta_path, "w") as f:
        json.dump(full_meta, f, indent=2, default=str)

    # Summary table
    print(f"\n\n{'=' * 70}")
    print("SUMMARY")
    print(f"{'=' * 70}")

    print(f"\n  {'Split':<10} {'FS':<16} {'Train':>6} {'Val':>6} {'Test':>6} "
          f"{'Tr%':>6} {'Va%':>6} {'Te%':>6}")
    print("  " + "-" * 70)
    for key, meta in sorted(all_metadata.items()):
        s, fs = key.split("/")
        print(f"  {s:<10} {fs:<16} {meta['train_rows']:>6,} {meta['val_rows']:>6,} "
              f"{meta['test_rows']:>6,} {meta['train_crash_rate']:>5.1%} "
              f"{meta['val_crash_rate']:>5.1%} {meta['test_crash_rate']:>5.1%}")

    n_files = sum(1 for _ in OUTPUT_DIR.rglob("*.parquet"))
    expected = len(SPLITS) * len(FEATURE_SETS) * 3
    print(f"\n  Files: {n_files} parquets (expected {expected})")

    if all_passed and n_files == expected:
        print(f"\n  ✓ ALL VALIDATIONS PASSED")
    else:
        print(f"\n  ✗ SOME VALIDATIONS FAILED")

    print(f"\n{'=' * 70}")
    print(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'=' * 70}")


# ═══════════════════════════════════════════════════════════════════════════════
# LOADER UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════

def load_split(split_name="Split_A", feature_set="full_moments", base_dir=OUTPUT_DIR):
    """Load a prepared split for model training."""
    split_dir = base_dir / split_name
    result = {}
    for part in ["train", "val", "test"]:
        df = pd.read_parquet(split_dir / f"{feature_set}_{part}.parquet")
        feature_cols = [c for c in df.columns if c not in META_COLS]
        result[f"X_{part}"] = df[feature_cols].to_numpy(dtype=np.float32)
        result[f"y_{part}"] = df["y_binary"].to_numpy(dtype=np.float32)
        result[f"minret_{part}"] = df["minret_5d_z"].to_numpy(dtype=np.float32)
        result[f"minret_raw_{part}"] = df["minret_5d"].to_numpy(dtype=np.float32)
        result[f"dates_{part}"] = df["date"].reset_index(drop=True)
        result[f"returns_{part}"] = df["target_daily_return"].to_numpy(dtype=np.float32)
    result["feature_cols"] = feature_cols
    meta_path = base_dir / "metadata.json"
    if meta_path.exists():
        with open(meta_path) as f:
            result["metadata"] = json.load(f)["splits"].get(f"{split_name}/{feature_set}", {})
    return result


def load_theme_assignment(feature_set="full_moments", base_dir=OUTPUT_DIR):
    """Load the corrected theme assignment CSV."""
    fnames = {
        "full_moments": "combined_full_moments_theme_assignment.csv",
        "means_only": "combined_means_theme_assignment.csv",
    }
    return pd.read_csv(base_dir / "themes" / fnames[feature_set])


if __name__ == "__main__":
    main()

DATASET PREPARATION: 4 Splits x 2 Feature Sets
Started: 2026-07-30 12:00:51

PHASE 0: FIX SUBTHEME IDs IN THEME ASSIGNMENT CSVs

  Fixing subtheme IDs in: combined_full_moments_theme_assignment.csv
  Subthemes before fix: 99
     1.10 -> 1_10  (Monthly Liquidity             ) : 17 features matched
     2.10 -> 2_10  (Trade Size & Timing           ) : 20 features matched
     3.10 -> 3_10  (VIX Futures & Term Structure  ) : 9 features matched
     8.10 -> 8_10  (Retail Sentiment Survey       ) : 5 features matched
     9.10 -> 9_10  (Bond Term Premium             ) : 4 features matched
    12.10 -> 12_10 (Surveys & Leading Indicators  ) : 6 features matched
  Total features reassigned: 61
  Subthemes after fix: 105
  All IDs converted to underscore format
    Moved 10 features to 2_11 (Monthly Volume Dynamics)
    ✓ 2_8 now has 45 features (daily only)
    ✓ 2_11 created with 10 features (monthly only)
    Moved 2 features to 12_11 (Weekly Unemployment Claims): ['initial_claims', 'conti

In [28]:
# %% [markdown]
# # Build Subtheme Update Schedule + Final Alignment Verification
#
# Run AFTER 01_prepare_datasets (uses df, theme_df, feature_cols from memory,
# OR can load from saved splits).
#
# Produces:
#   1. Verification that within each non-daily subtheme, every feature's
#      change-days are a subset of the representative's change-days.
#      This proves: features never update on a DIFFERENT day -- they either
#      update together or one is stale. Zero tolerance for misalignment.
#
#   2. An update mask DataFrame: (n_dates x n_non_daily_subthemes), True
#      on days when that subtheme received new data. This is what the
#      sparse KAN loads to know when to process each subtheme.
#
#   3. A summary CSV with subtheme_id, frequency, n_features,
#      n_update_days, representative_feature, update_day_pattern.

# %%
import pandas as pd
import numpy as np
from collections import defaultdict
from pathlib import Path

print("=" * 90)
print("SUBTHEME UPDATE SCHEDULE + FINAL ALIGNMENT CHECK")
print("=" * 90)

OUTPUT_DIR = Path("../../Data/Splits")
dates_ser = pd.to_datetime(df['date'])
n_days = len(df)

# ── Step 1: Build change-day sets for every feature ───────────────────────────
print("\nStep 1: Building change-day sets...")

change_days = {}  # col -> set of integer indices where value changed

for col in feature_cols:
    vals = df[col].values.astype(np.float64)
    indices = set()
    for i in range(1, n_days):
        if np.isnan(vals[i]) or np.isnan(vals[i - 1]):
            continue
        if abs(vals[i] - vals[i - 1]) > 1e-12:
            indices.add(i)
    change_days[col] = indices

print(f"  {len(change_days)} features processed.")

# ── Step 2: Build subtheme mapping with frequency ─────────────────────────────
subtheme_info = defaultdict(lambda: {'cols': [], 'name': '', 'freq': set()})

for _, row in theme_df.iterrows():
    col = str(row['column'])
    sid = str(row['subtheme_id'])
    sname = str(row['subtheme_name'])
    freq = str(row.get('frequency', 'daily'))
    if col in change_days:
        subtheme_info[sid]['cols'].append(col)
        subtheme_info[sid]['name'] = sname
        subtheme_info[sid]['freq'].add(freq)

# Non-daily subthemes only
non_daily_sids = [
    sid for sid, info in subtheme_info.items()
    if info['freq'] != {'daily'}
]

print(f"  Non-daily subthemes: {len(non_daily_sids)}")

# ── Step 3: For each non-daily subtheme, find representative + verify ─────────
print(f"\nStep 2: Verifying alignment and building update schedules...")

WEEKDAY_NAMES = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

schedule = {}        # sid -> set of update day indices
violations = {}      # sid -> list of violation details
summaries = []       # for the summary CSV

all_passed = True

for sid in sorted(non_daily_sids,
                  key=lambda x: (int(x.split('_')[0]), int(x.split('_')[1]))):
    info = subtheme_info[sid]
    cols = info['cols']
    sname = info['name']
    freq_str = '/'.join(sorted(info['freq']))

    if len(cols) == 0:
        continue

    # Representative = feature with the most change-days
    col_n_changes = {c: len(change_days[c]) for c in cols}
    rep_col = max(col_n_changes, key=col_n_changes.get)
    rep_days = change_days[rep_col]

    # Verify: every other feature's change-days must be a SUBSET of rep's
    # If feature X changed on a day the representative didn't change,
    # that's a genuine misalignment (not staleness).
    sub_violations = []

    for col in cols:
        if col == rep_col:
            continue
        col_days = change_days[col]
        not_in_rep = col_days - rep_days  # days col changed but rep didn't

        if not_in_rep:
            sub_violations.append({
                'feature': col,
                'n_orphan_days': len(not_in_rep),
                'example_dates': sorted(not_in_rep)[:5],
            })

    if sub_violations:
        violations[sid] = sub_violations
        all_passed = False

    # Record the schedule
    schedule[sid] = rep_days

    # Compute update day pattern -- different logic per frequency
    update_pattern = 'N/A'
    if rep_days:
        rep_timestamps = sorted([dates_ser.iloc[i] for i in rep_days])
        weekdays = [d.dayofweek for d in rep_timestamps]

        if 'weekly' in freq_str:
            # Weekly: modal weekday is meaningful (always same day)
            modal_wd = WEEKDAY_NAMES[pd.Series(weekdays).mode().iloc[0]]
            update_pattern = f"every {modal_wd}"
        elif 'monthly' in freq_str:
            # Monthly: show "~month-end" with typical day-of-month range
            days_of_month = [d.day for d in rep_timestamps]
            update_pattern = (f"month-end "
                             f"(day {min(days_of_month)}-{max(days_of_month)})")
        else:
            modal_wd = WEEKDAY_NAMES[pd.Series(weekdays).mode().iloc[0]]
            update_pattern = f"primary: {modal_wd}"

        # Also compute median gap between updates (trading days)
        if len(rep_timestamps) > 1:
            sorted_idx = sorted(rep_days)
            gaps = [sorted_idx[i+1] - sorted_idx[i] for i in range(len(sorted_idx)-1)]
            median_gap = int(np.median(gaps))
        else:
            median_gap = 0
    else:
        median_gap = 0

    summaries.append({
        'subtheme_id': sid,
        'subtheme_name': sname,
        'frequency': freq_str,
        'n_features': len(cols),
        'n_update_days': len(rep_days),
        'representative': rep_col,
        'update_pattern': update_pattern,
        'median_gap_days': median_gap,
    })

# ── Step 4: Print verification results ────────────────────────────────────────
print(f"\n{'=' * 90}")
print("ALIGNMENT VERIFICATION RESULTS")
print(f"{'=' * 90}")

print(f"\n  Non-daily subthemes checked: {len(non_daily_sids)}")
print(f"  Fully aligned: {len(non_daily_sids) - len(violations)}")
print(f"  With violations: {len(violations)}")

if violations:
    print(f"\n  VIOLATIONS:")
    for sid, viol_list in violations.items():
        sname = subtheme_info[sid]['name']
        print(f"\n  ✗ {sid} ({sname}):")
        for v in viol_list:
            example_dates = [str(dates_ser.iloc[i].date()) for i in v['example_dates']]
            print(f"    {v['feature']:<45} {v['n_orphan_days']} orphan days  "
                  f"e.g. {', '.join(example_dates)}")
else:
    print(f"\n  ✓ ALL non-daily subthemes are perfectly aligned.")
    print(f"    Every feature's change-days are a subset of its subtheme's")
    print(f"    representative. No feature ever updates on a different day")
    print(f"    from its peers.")

# ── Step 5: Print schedule summary ────────────────────────────────────────────
print(f"\n{'=' * 90}")
print("SUBTHEME UPDATE SCHEDULE")
print(f"{'=' * 90}")

print(f"\n  {'SubID':<10} {'Name':<40} {'Freq':<10} {'Feats':>6} "
      f"{'Updates':>8} {'Gap':>5} {'Update Pattern':<25} Representative")
print(f"  {'-' * 130}")

for s in sorted(summaries, key=lambda x: (int(x['subtheme_id'].split('_')[0]),
                                           int(x['subtheme_id'].split('_')[1]))):
    print(f"  {s['subtheme_id']:<10} {s['subtheme_name']:<40} {s['frequency']:<10} "
          f"{s['n_features']:>6} {s['n_update_days']:>8} {s['median_gap_days']:>5} "
          f"{s['update_pattern']:<25} {s['representative']}")

# ── Step 6: Build and save update mask ────────────────────────────────────────
print(f"\n{'=' * 90}")
print("BUILDING UPDATE MASK")
print(f"{'=' * 90}")

# Build mask: each date is a row, each non-daily subtheme is a column
# True = subtheme updated that day, False = no new data
mask_data = {'date': dates_ser.values}

for sid in sorted(non_daily_sids,
                  key=lambda x: (int(x.split('_')[0]), int(x.split('_')[1]))):
    col_name = f"update_{sid}"
    mask = np.zeros(n_days, dtype=bool)
    for idx in schedule.get(sid, set()):
        mask[idx] = True
    mask_data[col_name] = mask

update_mask = pd.DataFrame(mask_data)

# Daily subthemes are ALWAYS True (they update every day)
# We don't add them to the mask since the KAN processes them every day anyway

print(f"\n  Update mask shape: {update_mask.shape}")
print(f"  Non-daily subthemes in mask: {update_mask.shape[1] - 1}")
print(f"  Date range: {update_mask['date'].min()} -> {update_mask['date'].max()}")

# Stats per subtheme
print(f"\n  Updates per subtheme:")
for col in update_mask.columns:
    if col == 'date':
        continue
    n_true = update_mask[col].sum()
    pct = n_true / len(update_mask) * 100
    sid = col.replace('update_', '')
    sname = subtheme_info[sid]['name']
    print(f"    {sid:<10} {sname:<40} {n_true:>5} updates "
          f"({pct:.1f}% of trading days)")

# Save
mask_path = OUTPUT_DIR / "subtheme_update_mask.parquet"
update_mask.to_parquet(mask_path, index=False, engine='pyarrow')
print(f"\n  Saved: {mask_path}")

# Also save the summary CSV
summary_df = pd.DataFrame(summaries)
summary_path = OUTPUT_DIR / "subtheme_update_schedule.csv"
summary_df.to_csv(summary_path, index=False)
print(f"  Saved: {summary_path}")

# ── Step 7: Verify the mask is 100% correct against actual data ───────────────
print(f"\n{'=' * 90}")
print("MASK VERIFICATION -- cross-check against actual feature values")
print(f"{'=' * 90}")

# For each non-daily subtheme:
#   - Every day mask=True: at least one feature must have changed value
#   - Every day mask=False: NO feature should have changed value
# Zero tolerance.

mask_errors = 0
for sid in sorted(non_daily_sids,
                  key=lambda x: (int(x.split('_')[0]), int(x.split('_')[1]))):
    cols = subtheme_info[sid]['cols']
    sname = subtheme_info[sid]['name']
    mask_col = f"update_{sid}"

    if mask_col not in update_mask.columns:
        continue

    mask_true_days = set(np.where(update_mask[mask_col].values)[0])
    mask_false_days = set(range(n_days)) - mask_true_days

    # Union of all features' change-days in this subtheme
    all_change_days = set()
    for col in cols:
        all_change_days.update(change_days.get(col, set()))

    # Check 1: every mask=True day must have at least one feature that changed
    true_but_no_change = mask_true_days - all_change_days
    if true_but_no_change:
        example = sorted(true_but_no_change)[:3]
        example_dates = [str(dates_ser.iloc[i].date()) for i in example]
        print(f"  ✗ {sid} ({sname}): mask=True but no feature changed on "
              f"{len(true_but_no_change)} days, e.g. {', '.join(example_dates)}")
        mask_errors += 1

    # Check 2: every mask=False day must have NO feature that changed
    false_but_changed = mask_false_days & all_change_days
    if false_but_changed:
        example = sorted(false_but_changed)[:3]
        example_dates = [str(dates_ser.iloc[i].date()) for i in example]
        print(f"  ✗ {sid} ({sname}): mask=False but feature changed on "
              f"{len(false_but_changed)} days, e.g. {', '.join(example_dates)}")
        mask_errors += 1

if mask_errors == 0:
    print(f"\n  ✓ MASK IS PERFECTLY CORRECT")
    print(f"    Every True day has at least one feature that changed.")
    print(f"    Every False day has zero features that changed.")
    print(f"    Verified across {len(non_daily_sids)} subthemes x {n_days:,} trading days.")
else:
    print(f"\n  ✗ {mask_errors} mask errors found -- investigate above")

# ── Step 8: Quick sanity checks on the mask ───────────────────────────────────
print(f"\n{'=' * 90}")
print("SANITY CHECKS")
print(f"{'=' * 90}")

# Weekly subthemes should update ~once per 5 trading days (~20% of days)
# Monthly subthemes should update ~once per 21 trading days (~5% of days)
for s in summaries:
    pct = s['n_update_days'] / n_days * 100
    freq = s['frequency']
    sid = s['subtheme_id']

    if 'weekly' in freq and (pct < 10 or pct > 30):
        print(f"  ✗ {sid}: weekly but {pct:.1f}% update rate (expect 15-25%)")
    elif 'monthly' in freq and (pct < 1 or pct > 10):
        print(f"  ✗ {sid}: monthly but {pct:.1f}% update rate (expect 3-6%)")
    elif freq == 'daily' and pct < 90:
        print(f"  ✗ {sid}: daily but only {pct:.1f}% update rate")

print(f"\n  ✓ Schedule built and saved. The sparse KAN loads")
print(f"    subtheme_update_mask.parquet to know when to process each subtheme.")

SUBTHEME UPDATE SCHEDULE + FINAL ALIGNMENT CHECK

Step 1: Building change-day sets...
  2204 features processed.
  Non-daily subthemes: 52

Step 2: Verifying alignment and building update schedules...

ALIGNMENT VERIFICATION RESULTS

  Non-daily subthemes checked: 52
  Fully aligned: 52
  With violations: 0

  ✓ ALL non-daily subthemes are perfectly aligned.
    Every feature's change-days are a subset of its subtheme's
    representative. No feature ever updates on a different day
    from its peers.

SUBTHEME UPDATE SCHEDULE

  SubID      Name                                     Freq        Feats  Updates   Gap Update Pattern            Representative
  ----------------------------------------------------------------------------------------------------------------------------------
  1_10       Monthly Liquidity                        monthly        17      156    21 month-end (day 1-31)      monthly_BidAskSpread_cwmean
  2_11       Monthly Volume Dynamics                  monthly   

In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC: clip analysis — how many z-scores exceed 5 and 10, and per-feature.
# Reuses the attached pipeline's own functions/config. Loads and transforms
# in memory only; writes nothing.
# ═══════════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from pathlib import Path

# --- these come straight from the attached module's config ---
DATA_DIR = PROJECT_ROOT / "Data/Data_Collection/Final/Stage_4_Final_w_Calendar_Theme_Removed"
FS = FEATURE_SETS["full_moments"]     # test the 2204-feature set; repeat with "means_only" if wanted

# 1. Load Stage 4 source (same as Phase 2)
df = pd.read_parquet(DATA_DIR / FS["source_file"])
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]:,} cols")

# 2. Identify feature columns via the theme CSV, exactly as Phase 2 does
theme_df = pd.read_csv(OUTPUT_DIR / FS["theme_file"])   # the FIXED csv the pipeline writes
feature_cols = [c for c in theme_df["column"].values if c in df.columns]
print(f"Feature columns from theme CSV: {len(feature_cols)}")

# 3. Apply the SAME weekly z-score fix the pipeline applies before clipping
weekly_z_daily = build_weekly_zscored_daily(PANEL_C_PATH, WEEKLY_FEATURES)
df = replace_weekly_features(df, weekly_z_daily)

# 4. Restrict to the columns clip_zscores would actually clip (drop binary/meta)
clip_cols = [c for c in feature_cols if c not in DO_NOT_CLIP and c in df.columns]
print(f"Columns eligible for clipping (excl. binary/meta): {len(clip_cols)}\n")

# --- PER-FEATURE clip analysis (this is the part clip_zscores doesn't give you) ---
vals = df[clip_cols]
abs_max   = vals.abs().max()                       # per feature
n_gt5     = (vals.abs() > 5).sum()                 # per feature
n_gt10    = (vals.abs() > 10).sum()                # per feature
n_valid   = vals.notna().sum()                     # per feature (denominator)

summary = pd.DataFrame({
    "max_abs_z": abs_max,
    "n_gt5":  n_gt5,
    "n_gt10": n_gt10,
    "n_valid": n_valid,
}).sort_values("max_abs_z", ascending=False)

total_valid = int(n_valid.sum())
total_gt5   = int(n_gt5.sum())
total_gt10  = int(n_gt10.sum())

print("=" * 60)
print("GLOBAL")
print("=" * 60)
print(f"Global max |z|          : {abs_max.max():.2f}  ({abs_max.idxmax()})")
print(f"Features w/ any |z| > 5 : {int((abs_max > 5).sum())} / {len(clip_cols)}")
print(f"Features w/ any |z| >10 : {int((abs_max > 10).sum())} / {len(clip_cols)}")
print(f"Observations |z| > 5    : {total_gt5:,} / {total_valid:,} ({total_gt5/total_valid*100:.4f}%)")
print(f"Observations |z| >10    : {total_gt10:,} / {total_valid:,} ({total_gt10/total_valid*100:.4f}%)")

print("\n" + "=" * 60)
print("TOP 25 FEATURES BY MAX |z| (how they get clipped)")
print("=" * 60)
print(summary.head(25).to_string(
    formatters={"max_abs_z": "{:.2f}".format}))

# How the clip changes each of those columns: raw -> clipped-to-5
print("\n" + "=" * 60)
print("CLIP EFFECT on the single worst feature")
print("=" * 60)
worst = abs_max.idxmax()
w = df[worst].dropna()
print(f"Feature: {worst}")
print(f"  raw range        : [{w.min():.2f}, {w.max():.2f}]")
print(f"  values > +5      : {(w > 5).sum()}  (clipped down to +5)")
print(f"  values < -5      : {(w < -5).sum()}  (clipped up to -5)")
print(f"  values > +10     : {(w > 10).sum()}")
print(f"  values < -10     : {(w < -10).sum()}")

Loaded: 4,299 rows x 2,207 cols
Feature columns from theme CSV: 2204

  Building weekly-frequency z-scores from Panel C engineered...
    Source rows: 4,656  (2006-07-03 -> 2024-12-31)
    Weekly features: 34
    Min periods: 52
    1-day delay applied to: ['fed_assets', 'tga', 'reserves']

    Feature                    Updates  Valid days    First valid  Delayed
    ----------------------------------------------------------------------
    lev_long                       966       4,406     2007-07-02         
    lev_short                      966       4,406     2007-07-02         
    lev_spread                     966       4,406     2007-07-02         
    am_long                        966       4,406     2007-07-02         
    am_short                       966       4,406     2007-07-02         
    am_spread                      966       4,406     2007-07-02         
    dealer_long                    966       4,406     2007-07-02         
    dealer_short                 